<a href="https://colab.research.google.com/github/arnavm2k3/TorchCode-Solutions/blob/main/templates/10_gqa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.4 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [9]:
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        # Self-attention with grouped KVn
        B, seq_q, _ = x.size()

        Q = self.W_q(x).reshape(B, seq_q, self.num_heads, self.d_k).transpose(1, 2) # B x num_heads x seq_q x d_k
        K = self.W_k(x).reshape(B, seq_q, self.num_kv_heads, self.d_k).transpose(1, 2) # B x num_heads_kv x seq_q x d_k
        V = self.W_v(x).reshape(B, seq_q, self.num_kv_heads, self.d_k).transpose(1, 2) # B x num_heads_kv x seq_q x d_k

        K = torch.repeat_interleave(K, self.num_heads // self.num_kv_heads, dim=1) # B x num_heads x seq_q x d_k
        V = torch.repeat_interleave(V, self.num_heads // self.num_kv_heads, dim=1) # B x num_heads x seq_q x d_k

        attention = torch.matmul(torch.softmax(torch.matmul(Q, K.transpose(2, 3)) / math.sqrt(self.d_k), dim=-1), V) # B x num_heads x seq_q x d_k
        concat = attention.transpose(1, 2).contiguous().view(B, seq_q, self.d_model)
        mh_attention = self.W_o(concat)
        return mh_attention

In [10]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [11]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (4.3ms)
  ✅ [2/5] nn.Linear with correct shapes (1.0ms)
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (2.4ms)
  ✅ [4/5] KV heads are shared correctly (7.9ms)
  ✅ [5/5] Gradient flow (48.6ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (64.2ms total)
  Progress saved. Run status() to see your dashboard.

